In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_4.py ──
"""
# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 4: Shared Utilities
# ════════════════════════════════════════════════════════════════════════
#
# Common infrastructure for all Exercise 4 technique files:
#   - AG News loading + caching (120K train, 7.6K test)
#   - Vocabulary building and text-to-index conversion
#   - ExperimentTracker + ModelRegistry + OnnxBridge setup
#   - Shared constants, device detection, visualisation helpers
#
# ════════════════════════════════════════════════════════════════════════
"""

import asyncio
import math
from collections import Counter
from pathlib import Path
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset

import plotly.graph_objects as go

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml import OnnxBridge

# ════════════════════════════════════════════════════════════════════════
# Environment + Device
# ════════════════════════════════════════════════════════════════════════
setup_environment()

torch.manual_seed(42)
np.random.seed(42)
DEVICE = get_device()

# ════════════════════════════════════════════════════════════════════════
# Constants
# ════════════════════════════════════════════════════════════════════════
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
MAX_LEN = 40
VOCAB_SIZE = 15000
EPOCHS_SCRATCH = 8
BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LEN = 64
BERT_EPOCHS = 3
BERT_LR = 2e-5
BERT_BATCH_SIZE = 32


# ════════════════════════════════════════════════════════════════════════
# Core Attention Function (shared across 01, 02, 05)
# ════════════════════════════════════════════════════════════════════════
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot-product attention.

    Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V

    Args:
        q: Query tensor of shape (B, L_q, d_k)
        k: Key tensor of shape (B, L_k, d_k)
        v: Value tensor of shape (B, L_k, d_v)
        mask: Optional mask of shape (B, L_q, L_k). Positions with 0 are
              masked out (set to -inf before softmax).

    Returns:
        (output, attention_weights) where output has shape (B, L_q, d_v)
        and attention_weights has shape (B, L_q, L_k).
    """
    d_k = q.size(-1)
    scores = torch.einsum("bqd,bkd->bqk", q, k) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    out = torch.einsum("bqk,bkd->bqd", weights, v)
    return out, weights


# ════════════════════════════════════════════════════════════════════════
# AG News Loading
# ════════════════════════════════════════════════════════════════════════
def load_ag_news_split(split: str, cache_name: str) -> pl.DataFrame:
    """Load AG News split, caching to parquet for subsequent runs."""
    cache = Path.cwd() / "data" / "mlfp05" / cache_name
    cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        return pl.read_parquet(cache)
    print(f"  Downloading AG News {split} from HuggingFace...")
    ds = load_dataset("fancyzhx/ag_news", split=split)
    df = pl.from_pandas(ds.to_pandas())
    df.write_parquet(cache)
    return df


def load_ag_news() -> tuple[pl.DataFrame, pl.DataFrame]:
    """Load full AG News train + test splits with status reporting."""
    print("\n== Loading FULL AG News (120K train + 7.6K test) ==")
    train_df = load_ag_news_split("train", "ag_news_full_train.parquet")
    test_df = load_ag_news_split("test", "ag_news_full_test.parquet")
    print(f"  train rows: {len(train_df):,}   test rows: {len(test_df):,}")
    print(f"  sample headline: {train_df['text'][0][:80]!r}")
    print(f"  class balance (train): {dict(Counter(train_df['label'].to_list()))}")
    return train_df, test_df


# ════════════════════════════════════════════════════════════════════════
# Vocabulary + Tokenisation (for from-scratch models: LSTM + Transformer)
# ════════════════════════════════════════════════════════════════════════
def build_vocab(texts: list[str], max_vocab: int = VOCAB_SIZE) -> dict[str, int]:
    """Build a word-level vocabulary from text list."""
    words: Counter[str] = Counter()
    for t in texts:
        words.update(t.lower().split())
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, _ in words.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab


def text_to_indices(
    text: str, vocab: dict[str, int], max_len: int = MAX_LEN
) -> list[int]:
    """Convert text to padded index sequence."""
    tokens = text.lower().split()[:max_len]
    idxs = [vocab.get(t, 1) for t in tokens]
    return (idxs + [0] * (max_len - len(idxs)))[:max_len]


def prepare_dataloaders(
    train_df: pl.DataFrame,
    test_df: pl.DataFrame,
    vocab: dict[str, int],
    batch_size: int = 128,
) -> tuple[
    DataLoader, DataLoader, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor
]:
    """Tokenise AG News and build DataLoaders for from-scratch models.

    Returns: (train_loader, val_loader, train_t, train_y, test_t, test_y)
    """
    train_tokens = np.array(
        [text_to_indices(t, vocab, MAX_LEN) for t in train_df["text"].to_list()],
        dtype=np.int64,
    )
    train_labels = np.array(train_df["label"].to_list(), dtype=np.int64)
    test_tokens = np.array(
        [text_to_indices(t, vocab, MAX_LEN) for t in test_df["text"].to_list()],
        dtype=np.int64,
    )
    test_labels = np.array(test_df["label"].to_list(), dtype=np.int64)

    train_t = torch.from_numpy(train_tokens).to(DEVICE)
    train_y = torch.from_numpy(train_labels).to(DEVICE)
    test_t = torch.from_numpy(test_tokens).to(DEVICE)
    test_y = torch.from_numpy(test_labels).to(DEVICE)

    train_loader = DataLoader(
        TensorDataset(train_t, train_y), batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(TensorDataset(test_t, test_y), batch_size=batch_size)

    return train_loader, val_loader, train_t, train_y, test_t, test_y


# ════════════════════════════════════════════════════════════════════════
# Kailash-ML Engine Setup
# ════════════════════════════════════════════════════════════════════════
async def _setup_engines_async() -> (
    tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]
):
    """Initialise ExperimentTracker (kailash-ml 1.1.1 factory) + ModelRegistry."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_transformers.db"
    registry_db = "sqlite:///mlfp05_transformers_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_transformers", registry, True


def setup_engines() -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
    OnnxBridge,
]:
    """Set up all kailash-ml engines (sync wrapper).

    Returns: (conn, tracker, exp_name, registry, has_registry, bridge)
    """
    conn, tracker, exp_name, registry, has_registry = asyncio.run(
        _setup_engines_async()
    )
    bridge = OnnxBridge()
    return conn, tracker, exp_name, registry, has_registry, bridge


# ════════════════════════════════════════════════════════════════════════
# Training Utilities
# ════════════════════════════════════════════════════════════════════════
async def train_model_async(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = EPOCHS_SCRATCH,
    lr: float = 2e-3,
) -> tuple[list[float], list[float]]:
    """Train a PyTorch classifier and log every epoch to ExperimentTracker.

    Uses the modern ``tracker.track(...)`` async context manager -- bulk
    param logging, per-step metrics, automatic COMPLETED/FAILED state.
    """
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    train_losses: list[float] = []
    val_accs: list[float] = []
    best_acc = 0.0
    best_state: dict | None = None
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    async with tracker.track(experiment=exp_name, run_name=model_name) as run:
        await run.log_params(
            {
                "model_type": model_name,
                "epochs": str(epochs),
                "lr": str(lr),
                "dataset_size": str(len(train_loader.dataset)),
                "batch_size": str(train_loader.batch_size),
                "trainable_params": str(param_count),
            }
        )

        for epoch in range(epochs):
            model.train()
            batch_losses = []
            for xb, yb in train_loader:
                optimizer.zero_grad()
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                batch_losses.append(loss.item())
            scheduler.step()
            epoch_loss = float(np.mean(batch_losses))
            train_losses.append(epoch_loss)

            model.eval()
            with torch.no_grad():
                correct = 0
                total = 0
                for xb, yb in val_loader:
                    preds = model(xb).argmax(dim=-1)
                    correct += int((preds == yb).sum().item())
                    total += int(yb.size(0))
                acc = correct / total
                val_accs.append(acc)

            await run.log_metrics(
                {"train_loss": epoch_loss, "val_accuracy": acc},
                step=epoch + 1,
            )
            if acc > best_acc:
                best_acc = acc
                best_state = {
                    k: v.detach().clone() for k, v in model.state_dict().items()
                }
            print(
                f"  [{model_name}] epoch {epoch+1}/{epochs}  "
                f"loss={epoch_loss:.4f}  val_acc={acc:.3f}"
            )

        await run.log_metrics(
            {
                "best_val_accuracy": best_acc,
                "final_train_loss": train_losses[-1],
            }
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return train_losses, val_accs


def train_model(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = EPOCHS_SCRATCH,
    lr: float = 2e-3,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model, model_name, train_loader, val_loader, tracker, exp_name, epochs, lr
        )
    )


# ════════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ════════════════════════════════════════════════════════════════════════
def create_attention_heatmap(
    attn_weights: np.ndarray,
    labels: list[str],
    title: str = "Self-Attention Heatmap",
    max_tokens: int = 15,
) -> go.Figure:
    """Create a plotly heatmap from an attention weight matrix.

    Args:
        attn_weights: 2D numpy array of shape (seq, seq)
        labels: token labels for axes
        title: chart title
        max_tokens: limit display to first N non-pad tokens

    Returns:
        plotly Figure
    """
    show_len = min(max_tokens, len([w for w in labels if w != "<pad>"]))
    fig = go.Figure(
        data=go.Heatmap(
            z=attn_weights[:show_len, :show_len],
            x=labels[:show_len],
            y=labels[:show_len],
            colorscale="Viridis",
            colorbar={"title": "Attention weight"},
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Key position",
        yaxis_title="Query position",
        width=600,
        height=500,
    )
    return fig


def get_viz() -> ModelVisualizer:
    """Return a ModelVisualizer instance."""
    return ModelVisualizer()




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 4.3: LSTM Baseline for Comparison
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Explain why baselines are essential for evaluating new architectures
#   - Build a bidirectional LSTM text classifier for fair comparison
#   - Contrast sequential (LSTM) vs parallel (Transformer) processing
#   - Train the LSTM on the same data and compare training dynamics
#   - Apply LSTM classification to a Singapore airline customer feedback use case
#
# PREREQUISITES: ex_4/01_self_attention_from_scratch.py
# ESTIMATED TIME: ~20 min
# DATASET: AG News — 120,000 real news headlines, 4 classes.
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


print(f"Using device: {DEVICE}")



## THEORY — Why We Need a Baseline


In [ ]:
# When evaluating a new architecture (like the Transformer), you need a
# fair comparison point. Without a baseline, you cannot tell whether the
# Transformer's accuracy comes from:
#   (a) The attention mechanism itself, or
#   (b) Having more parameters, better hyperparameters, or more training.
#
# The LSTM is the strongest pre-Transformer baseline for text. It processes
# tokens sequentially, maintaining a hidden state that accumulates context.
# The key differences:
#
#   LSTM (sequential):
#     - Processes tokens one at a time: O(n) sequential steps
#     - Information flows through a hidden state "bottleneck"
#     - Bidirectional LSTM reads forward AND backward, doubling the context
#     - Cannot be parallelised across positions during training
#
#   Transformer (parallel):
#     - Processes all tokens simultaneously: O(1) depth (but O(n^2) attention)
#     - Every token directly attends to every other token
#     - Positional encoding provides word order information
#     - Fully parallelisable during training (much faster on GPU)
#
# By training both on the same data with similar parameter counts, we
# isolate the architectural difference and measure what attention buys.



## TASK 1 — Load data and set up engines


In [ ]:
train_df, test_df = load_ag_news()
vocab = build_vocab(train_df["text"].to_list())
train_loader, val_loader, train_t, train_y, test_t, test_y = prepare_dataloaders(
    train_df, test_df, vocab
)
conn, tracker, exp_name, registry, has_registry, bridge = setup_engines()
print(f"  vocab size: {len(vocab)}, seq_len: {MAX_LEN}")



## TASK 2 — Build: Bidirectional LSTM Text Classifier


Architecture: embedding -> bidirectional LSTM -> mean pool -> classifier.

    Bidirectional processing reads the sequence both forward and backward,
    giving each token context from both directions. This partially addresses
    the LSTM's sequential limitation, but information must still propagate
    through the hidden state chain -- unlike attention, which is direct.


In [ ]:
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        hidden_dim: int = 128,
        n_layers: int = 2,
        n_classes: int = 4,
        dropout: float = 0.3,
    ):
        super().__init__()
        # TODO: Build the LSTM architecture
        # Hint: self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Hint: self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
        #              batch_first=True, dropout=dropout if n_layers > 1 else 0.0,
        #              bidirectional=True)
        # Hint: self.head_drop = nn.Dropout(dropout)
        # Hint: self.head = nn.Linear(hidden_dim * 2, n_classes)  — *2 because bidirectional
        self.embed = ...  # YOUR CODE HERE
        self.lstm = ...  # YOUR CODE HERE
        self.head_drop = ...  # YOUR CODE HERE
        self.head = ...  # YOUR CODE HERE

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        # TODO: Implement forward pass
        # Step 1: x = self.embed(tokens)
        # Step 2: lstm_out, _ = self.lstm(x)  — shape (B, L, 2*H)
        # Step 3: Mean pool over non-pad positions
        #   pad_mask = (tokens == 0)
        #   lengths = (~pad_mask).sum(dim=1, keepdim=True).clamp(min=1).float()
        #   lstm_out = lstm_out.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        #   pooled = lstm_out.sum(dim=1) / lengths
        # Step 4: return self.head(self.head_drop(pooled))
        ...  # YOUR CODE HERE



### Checkpoint 1


In [ ]:
lstm_test = LSTMClassifier(vocab_size=100).to(DEVICE)
dummy_tokens = torch.randint(0, 100, (2, MAX_LEN), device=DEVICE)
lstm_out = lstm_test(dummy_tokens)
assert lstm_out.shape == (2, 4), "LSTM should output (batch, 4 classes)"
param_count = sum(p.numel() for p in lstm_test.parameters())
print(f"\n  LSTMClassifier output shape: {tuple(lstm_out.shape)} (expected (2, 4))")
print(f"  Parameter count: {param_count:,}")
print("\n--- Checkpoint 1 passed --- LSTM architecture ready\n")



## TASK 3 — Train: LSTM on full AG News with ExperimentTracker


In [ ]:
print("\n== Training LSTM baseline on full AG News ==")
# TODO: Create LSTMClassifier and train it
# Hint: lstm_model = LSTMClassifier(vocab_size=len(vocab), embed_dim=128, hidden_dim=128, n_layers=2, n_classes=4)
# Hint: lstm_losses, lstm_accs = train_model(lstm_model, "lstm_baseline",
#           train_loader, val_loader, tracker, exp_name, epochs=EPOCHS_SCRATCH)
lstm_model = ...  # YOUR CODE HERE
lstm_losses, lstm_accs = ...  # YOUR CODE HERE



### Checkpoint 2


In [ ]:
assert len(lstm_losses) == EPOCHS_SCRATCH, "LSTM should train for all epochs"
assert (
    max(lstm_accs) > 0.60
), f"LSTM should reach >60% accuracy, got {max(lstm_accs):.3f}"
# INTERPRETATION: The LSTM provides a strong baseline. On short headlines
# (avg ~10 words), the LSTM's sequential bottleneck isn't as severe as it
# would be on longer documents. The real gap between LSTM and Transformer
# widens as sequence length increases -- on 512-token documents, the
# Transformer's direct attention outperforms LSTM by a wider margin.
print(f"\n  LSTM best accuracy: {max(lstm_accs):.3f}")
print(f"  LSTM final loss: {lstm_losses[-1]:.4f}")
print("\n--- Checkpoint 2 passed --- LSTM baseline trained\n")



## TASK 4 — Visualise: LSTM training dynamics


In [ ]:
# The LSTM's training curve reveals its learning dynamics. Unlike the
# Transformer, which can leverage parallel attention from epoch 1, the
# LSTM must learn to propagate information through the hidden state chain.

# TODO: Visualise LSTM training curves using ModelVisualizer
# Hint: from shared.mlfp05.ex_4 import get_viz
# Hint: viz = get_viz()
# Hint: fig_lstm = viz.training_history(metrics={"LSTM train_loss": lstm_losses, "LSTM val_accuracy": lstm_accs}, x_label="Epoch", y_label="Value")
# Hint: fig_lstm.write_html("ex_4_3_lstm_training_curves.html")

viz = ...  # YOUR CODE HERE — get_viz()
fig_lstm = ...  # YOUR CODE HERE — viz.training_history(...)
...  # YOUR CODE HERE — fig_lstm.write_html("ex_4_3_lstm_training_curves.html")
print("  LSTM training curves saved to ex_4_3_lstm_training_curves.html")



### Checkpoint 3


In [ ]:
assert len(lstm_accs) == EPOCHS_SCRATCH, "Should have accuracy for each epoch"
# INTERPRETATION: The LSTM learning curve typically shows:
#   - Rapid initial learning (epochs 1-3): learns common word-class associations
#   - Slower improvement (epochs 4-6): refining contextual understanding
#   - Plateau (epochs 7-8): limited by the sequential bottleneck
# The Transformer, by contrast, often shows faster initial convergence
# because attention provides immediate access to all positions.
print("\n--- Checkpoint 3 passed --- LSTM training dynamics visualised\n")



## TASK 5 — Apply: Customer Feedback Classification for Singapore Airlines


In [ ]:
# SCENARIO: Singapore Airlines (SQ) collects thousands of customer reviews
# monthly across Skytrax, Google Reviews, social media, and their own
# feedback portal. The customer experience team needs to classify each
# review by topic -- service, food, seat comfort, delays -- to route it
# to the right operational team for action.
#
# BUSINESS VALUE: Manual review classification takes 2-3 minutes per review.
# With ~5,000 reviews/month, that is 167-250 hours of analyst time. An
# automated LSTM classifier handles the first-pass routing, letting analysts
# focus on extracting actionable insights rather than sorting.
#
# WHY LSTM HERE: The LSTM baseline establishes the accuracy floor. If the
# LSTM achieves 85% routing accuracy, the Transformer needs to beat that
# to justify its higher computational cost. If the LSTM achieves 95%, the
# Transformer's marginal improvement may not justify the infrastructure
# investment. This is the fundamental question baselines answer.
#
# SPEED COMPARISON: On a single GPU, the LSTM processes ~2,000 reviews/second
# (sequential processing). The Transformer processes ~5,000 reviews/second
# (parallel attention). For 5,000 reviews/month, both are fast enough --
# the speed difference matters at scale (millions of reviews).
print("\n== Application: Customer Feedback at Singapore Airlines ==")

sq_reviews = [
    "World class service from cabin crew on long haul flight",
    "New business class seat design wins innovation award",
    "Flight delayed three hours due to technical issues at Changi",
    "Award winning food menu designed by celebrity chef",
    "Technology upgrade to in-flight entertainment system completed",
]
review_topics = ["Service", "Seat", "Delay", "Food", "Tech"]

# TODO: Classify reviews with the trained LSTM model
# Step 1: lstm_model.eval()
# Step 2: sq_idx = torch.tensor([text_to_indices(t, vocab, MAX_LEN) for t in sq_reviews], dtype=torch.long, device=DEVICE)
# Step 3: with torch.no_grad(): get logits, probs, preds
#   sq_logits = lstm_model(sq_idx)
#   sq_probs = F.softmax(sq_logits, dim=-1)
#   sq_preds = sq_logits.argmax(dim=-1).cpu().tolist()
lstm_model.eval()
with torch.no_grad():
    sq_idx = ...  # YOUR CODE HERE
    sq_logits = ...  # YOUR CODE HERE
    sq_probs = ...  # YOUR CODE HERE
    sq_preds = ...  # YOUR CODE HERE

print(f"\n  Singapore Airlines review classification (LSTM baseline):")
print(f"  {'Review':<55} {'Topic':<8} {'AG News Class':<12} {'Confidence':>10}")
print("  " + "-" * 87)
for text, topic, pred, probs in zip(
    sq_reviews, review_topics, sq_preds, sq_probs.cpu().tolist()
):
    cls_name = CLASS_NAMES[pred]
    confidence = max(probs)
    print(f"  {text[:53]:<55} {topic:<8} {cls_name:<12} {confidence:>10.1%}")

# TODO: Measure throughput
# Hint: import time
# Hint: batch_input = torch.randint(0, len(vocab), (128, MAX_LEN), device=DEVICE)
# Hint: with torch.no_grad(): run 10 batches, measure time, compute throughput
import time

batch_input = torch.randint(0, len(vocab), (128, MAX_LEN), device=DEVICE)
with torch.no_grad():
    t0 = time.perf_counter()
    for _ in range(10):
        _ = lstm_model(batch_input)
    t1 = time.perf_counter()
    throughput = ...  # YOUR CODE HERE — (128 * 10) / (t1 - t0)
    print(f"\n  LSTM throughput: {throughput:,.0f} reviews/second")



### Checkpoint 4


In [ ]:
assert len(sq_preds) == len(sq_reviews), "Should classify all reviews"
# INTERPRETATION: The LSTM provides a solid baseline for customer feedback
# classification. Even without domain-specific training, it captures
# topic-relevant patterns in text. The Transformer (next exercise) will
# typically match or exceed this accuracy while processing reviews faster.
#
# BUSINESS IMPACT for Singapore Airlines:
#   - 5,000 customer reviews/month
#   - 2-3 min manual classification per review -> seconds with LSTM
#   - Annual saving: 2,000-3,000 analyst hours
#   - Faster issue escalation: delay complaints reach operations within minutes
#   - The Transformer comparison (next) determines if the accuracy uplift
#     justifies the additional compute cost
print("\n--- Checkpoint 4 passed --- Singapore Airlines application complete\n")



## REFLECTION


[x] Understood why baselines are essential for fair evaluation
  [x] Built a bidirectional LSTM text classifier
  [x] Contrasted sequential (LSTM) vs parallel (Transformer) processing
  [x] Trained on full AG News (120K headlines), best acc: {max(lstm_accs):.1%}
  [x] Measured inference throughput for production sizing
  [x] Applied to Singapore Airlines customer feedback classification

  KEY INSIGHT:
    The LSTM is a strong baseline, not a strawman. On short sequences
    (headlines, tweets), it's competitive with transformers. The gap
    widens on longer documents where the sequential bottleneck hurts.
    Always establish your baseline BEFORE claiming your new architecture
    is "better" -- the margin matters more than the absolute number.

  Next: In 04_bert_finetuning.py, you'll see what happens when you
  combine the Transformer architecture with massive pre-training.
  BERT doesn't just learn from your 120K headlines -- it brings
  knowledge from billions of words of pre-training.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — LSTM Baseline")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — LSTM baseline (contrast with Transformer 02)


In [ ]:
from kailash_ml import diagnose

print("\n── Diagnostic Report (LSTM baseline) ──")
report = diagnose(lstm_model, kind="dl", data=val_loader, show=False)
# ══════ EXPECTED OUTPUT (reference pattern — LSTM on AG News) ═══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
#   [!] Gradient flow (WARNING): RMS ratio
#       `lstm.weight_ih_l0` : `head.weight` ≈ 1:50 —
#       gradients concentrated at the classifier head, modest
#       signal reaching the embedding layer. Classic mild form
#       of the vanishing-gradient-through-time problem.
#   [✓] Activations    (HEALTHY): tanh/sigmoid gates within
#       [-1, 1] and [0, 1] — no saturation.
#   [✓] Loss trend     (HEALTHY): train loss decreases steadily;
#       val acc plateaus ~0.86 by epoch 6.



## Best val acc: ~0.86 after 8 epochs — within 2% of the Transformer
on AG News (headlines are short, so the LSTM's sequential
bottleneck is manageable).

STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:

 [BLOOD TEST] Gradients concentrate in the CLASSIFIER HEAD.
    The LSTM's recurrent weights receive ~50x less signal than
    `head.weight`. On 25-token headlines this is tolerable; on
    500-token documents the same architecture would collapse
    (the first 400 tokens' embeddings never update). This IS
    the "vanishing gradient through time" that slide 5.3 warns
    about — the Transformer sidesteps it entirely via direct
    attention + residual connections.
    >> Prescription Pad: if deploying on long documents, switch
       to Transformer (ex_4/02) or add gradient clipping +
       truncated BPTT. Bidirectional helps but doesn't eliminate
       the issue.

 [X-RAY] LSTM gate activations healthy — sigmoid outputs in
    [0, 1] without saturating at either end. Saturation would
    mean the forget gate is stuck "always remember" or "always
    forget" which kills the LSTM's ability to learn what to
    keep. If the X-Ray flags this, lower the learning rate and
    add LayerNorm inside the LSTM cell.
    >> Prescription Pad: healthy — no action.

 [STETHOSCOPE] Loss curve shows the classic LSTM training
    shape: rapid descent for 3 epochs, then a plateau as the
    model hits the sequential-bottleneck ceiling. The
    Transformer (02) shows a similar trajectory but reaches
    a lower plateau — its plateau is the "architectural
    ceiling" of the data, not the model.
    >> Prescription Pad: compare final val acc side-by-side
       with the Transformer to quantify the headroom.

 FIVE-INSTRUMENT TAKEAWAY: on SHORT text the LSTM is 95%
 as good as the Transformer. The Prescription Pad shows WHY
 you would still prefer the Transformer — the gradient-flow
 WARNING will become CRITICAL on real production documents
 (news articles, support tickets, legal briefs). Never choose
 an architecture on toy-length inputs; profile on your real
 sequence length distribution.

 CONNECT TO SLIDE 5.3 (RNNs) + 5.4 (Transformers): slide 5.3's
 unrolled-RNN diagram shows information squeezing through the
 hidden state; the WARNING reading above is the quantitative
 version of that diagram. Slide 5.4 claims attention "lets
 every token reach every other token in one step" — compare
 this WARNING with the HEALTHY Blood Test in ex_4/02 to see
 the architectural payoff in numbers.


### Checkpoint 2


In [ ]:
assert len(lstm_losses) == EPOCHS_SCRATCH, "LSTM should train for all epochs"
assert (
    max(lstm_accs) > 0.60
), f"LSTM should reach >60% accuracy, got {max(lstm_accs):.3f}"
# INTERPRETATION: The LSTM provides a strong baseline. On short headlines
# (avg ~10 words), the LSTM's sequential bottleneck isn't as severe as it
# would be on longer documents. The real gap between LSTM and Transformer
# widens as sequence length increases -- on 512-token documents, the
# Transformer's direct attention outperforms LSTM by a wider margin.
print(f"\n  LSTM best accuracy: {max(lstm_accs):.3f}")
print(f"  LSTM final loss: {lstm_losses[-1]:.4f}")
print("\n--- Checkpoint 2 passed --- LSTM baseline trained\n")



## TASK 4 — Visualise: LSTM training dynamics


In [ ]:
# The LSTM's training curve reveals its learning dynamics. Unlike the
# Transformer, which can leverage parallel attention from epoch 1, the
# LSTM must learn to propagate information through the hidden state chain.

viz = get_viz()
fig_lstm = viz.training_history(
    metrics={
        "LSTM train_loss": lstm_losses,
        "LSTM val_accuracy": lstm_accs,
    },
    x_label="Epoch",
    y_label="Value",
)
fig_lstm.write_html("ex_4_3_lstm_training_curves.html")
print("  LSTM training curves saved to ex_4_3_lstm_training_curves.html")



### Checkpoint 3


In [ ]:
assert len(lstm_accs) == EPOCHS_SCRATCH, "Should have accuracy for each epoch"
# INTERPRETATION: The LSTM learning curve typically shows:
#   - Rapid initial learning (epochs 1-3): learns common word-class associations
#   - Slower improvement (epochs 4-6): refining contextual understanding
#   - Plateau (epochs 7-8): limited by the sequential bottleneck
# The Transformer, by contrast, often shows faster initial convergence
# because attention provides immediate access to all positions.
print("\n--- Checkpoint 3 passed --- LSTM training dynamics visualised\n")



## TASK 5 — Apply: Customer Feedback Classification for Singapore Airlines


In [ ]:
# SCENARIO: Singapore Airlines (SQ) collects thousands of customer reviews
# monthly across Skytrax, Google Reviews, social media, and their own
# feedback portal. The customer experience team needs to classify each
# review by topic -- service, food, seat comfort, delays -- to route it
# to the right operational team for action.
#
# BUSINESS VALUE: Manual review classification takes 2-3 minutes per review.
# With ~5,000 reviews/month, that is 167-250 hours of analyst time. An
# automated LSTM classifier handles the first-pass routing, letting analysts
# focus on extracting actionable insights rather than sorting.
#
# WHY LSTM HERE: The LSTM baseline establishes the accuracy floor. If the
# LSTM achieves 85% routing accuracy, the Transformer needs to beat that
# to justify its higher computational cost. If the LSTM achieves 95%, the
# Transformer's marginal improvement may not justify the infrastructure
# investment. This is the fundamental question baselines answer.
#
# SPEED COMPARISON: On a single GPU, the LSTM processes ~2,000 reviews/second
# (sequential processing). The Transformer processes ~5,000 reviews/second
# (parallel attention). For 5,000 reviews/month, both are fast enough --
# the speed difference matters at scale (millions of reviews).
print("\n== Application: Customer Feedback at Singapore Airlines ==")

# Classify sample "customer reviews" (using AG News as proxy data)
sq_reviews = [
    "World class service from cabin crew on long haul flight",
    "New business class seat design wins innovation award",
    "Flight delayed three hours due to technical issues at Changi",
    "Award winning food menu designed by celebrity chef",
    "Technology upgrade to in-flight entertainment system completed",
]
review_topics = ["Service", "Seat", "Delay", "Food", "Tech"]

lstm_model.eval()
with torch.no_grad():
    sq_idx = torch.tensor(
        [text_to_indices(t, vocab, MAX_LEN) for t in sq_reviews],
        dtype=torch.long,
        device=DEVICE,
    )
    sq_logits = lstm_model(sq_idx)
    sq_probs = F.softmax(sq_logits, dim=-1)
    sq_preds = sq_logits.argmax(dim=-1).cpu().tolist()

print(f"\n  Singapore Airlines review classification (LSTM baseline):")
print(f"  {'Review':<55} {'Topic':<8} {'AG News Class':<12} {'Confidence':>10}")
print("  " + "-" * 87)
for text, topic, pred, probs in zip(
    sq_reviews, review_topics, sq_preds, sq_probs.cpu().tolist()
):
    cls_name = CLASS_NAMES[pred]
    confidence = max(probs)
    print(f"  {text[:53]:<55} {topic:<8} {cls_name:<12} {confidence:>10.1%}")

# Measure throughput
import time

lstm_model.eval()
batch_input = torch.randint(0, len(vocab), (128, MAX_LEN), device=DEVICE)
with torch.no_grad():
    t0 = time.perf_counter()
    for _ in range(10):
        _ = lstm_model(batch_input)
    t1 = time.perf_counter()
    throughput = (128 * 10) / (t1 - t0)
    print(f"\n  LSTM throughput: {throughput:,.0f} reviews/second")



### Checkpoint 4


In [ ]:
assert len(sq_preds) == len(sq_reviews), "Should classify all reviews"
# INTERPRETATION: The LSTM provides a solid baseline for customer feedback
# classification. Even without domain-specific training, it captures
# topic-relevant patterns in text. The Transformer (next exercise) will
# typically match or exceed this accuracy while processing reviews faster.
#
# BUSINESS IMPACT for Singapore Airlines:
#   - 5,000 customer reviews/month
#   - 2-3 min manual classification per review -> seconds with LSTM
#   - Annual saving: 2,000-3,000 analyst hours
#   - Faster issue escalation: delay complaints reach operations within minutes
#   - The Transformer comparison (next) determines if the accuracy uplift
#     justifies the additional compute cost
print("\n--- Checkpoint 4 passed --- Singapore Airlines application complete\n")



## REFLECTION


[x] Understood why baselines are essential for fair evaluation
  [x] Built a bidirectional LSTM text classifier
  [x] Contrasted sequential (LSTM) vs parallel (Transformer) processing
  [x] Trained on full AG News (120K headlines), best acc: {max(lstm_accs):.1%}
  [x] Measured inference throughput for production sizing
  [x] Applied to Singapore Airlines customer feedback classification

  KEY INSIGHT:
    The LSTM is a strong baseline, not a strawman. On short sequences
    (headlines, tweets), it's competitive with transformers. The gap
    widens on longer documents where the sequential bottleneck hurts.
    Always establish your baseline BEFORE claiming your new architecture
    is "better" -- the margin matters more than the absolute number.

  Next: In 04_bert_finetuning.py, you'll see what happens when you
  combine the Transformer architecture with massive pre-training.
  BERT doesn't just learn from your 120K headlines -- it brings
  knowledge from billions of words of pre-training.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — LSTM Baseline")
print("=" * 70)
print(
)

